# 06. Backdoor Trojan and Poisoning Defense: Neural Cleanse and Spectral Signatures

- **Category:** 11_ai_safety_privacy_and_verification
- **Lineage & Citations:** Gu et al. (2017 *BadNets: Identifying Vulnerabilities in the Machine Learning Model Supply Chain*); Tran et al. (2018 *Spectral Signatures in Backdoor Attacks*); Wang et al. (2019 *Neural Cleanse: Identifying and Mitigating Backdoor Attacks in Neural Networks*); Gao et al. (2019 *STRIP: A Defence Against Trojan Attacks on Deep Neural Networks*).
- **Purpose:** Formulate and evaluate backdoor trojan attacks and multi-tier defenses: Spectral Signature SVD covariance filtering, Neural Cleanse trigger inversion with Median Absolute Deviation (MAD) anomaly testing, and STRIP runtime Shannon entropy detection.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [05. Safe RL Constrained MDP PPO Lagrangian CVaR and CBF](05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.ipynb) $\to$ `🔗 Downstream Syntheses:` [07. Privacy DP SGD PATE Gradient Inversion DLG and MIA](07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not rely on standard fine-tuning to purge backdoors (trojan neurons remain dormant on clean data and survive fine-tuning without Fine-Pruning); do not use naive Euclidean distance filtering instead of representation covariance SVD.

## Mathematical Foundations & Governing Equations

1. **Spectral Signature Outlier Score**: $\text{score}_i = \left( (h_i - \mu) \cdot v_1 \right)^2, \quad \Sigma v_1 = \lambda_{\max} v_1$
2. **Neural Cleanse MAD Anomaly**: $\text{Anomaly}_c = \frac{\text{median}(S) - S_c}{1.4826 \cdot \text{MAD}} > 2.0$
3. **STRIP Runtime Entropy**: $\bar{H} = -\frac{1}{K} \sum_{k=1}^K \sum_c P_k(c) \ln(P_k(c) + \epsilon) \approx 0$
4. **BadNets Trigger Injection**: $x' = x \odot (1 - m) + p \odot m$

In [ ]:
import sys, time
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/11_ai_safety_privacy_and_verification/06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_spectral(reprs):
    mu = torch.mean(reprs, dim=0, keepdim=True)
    centered = reprs - mu
    cov = (centered.t() @ centered) / float(reprs.shape[0])
    d = cov.shape[0]
    v = torch.ones(d) / (float(d) ** 0.5)
    for _ in range(30):
        v = cov @ v
        v = v / (torch.norm(v, p=2) + 1e-8)
    return (centered @ v) ** 2

def calc_cleanse_mad(l1_norms):
    med = torch.median(l1_norms)
    mad = torch.median(torch.abs(l1_norms - med)) + 1e-8
    return (med - l1_norms) / (1.4826 * mad)

def calc_strip(logits):
    probs = F.softmax(logits, dim=-1)
    return torch.mean(-torch.sum(probs * torch.log(probs + 1e-12), dim=-1))

def apply_trigger(x, mask, pattern):
    return x * (1.0 - mask) + pattern * mask

In [ ]:
# Parity verification against golden reference tensors
spec_scores = calc_spectral(weights["representations"])
assert torch.max(torch.abs(spec_scores - weights["spectral_scores_ref"])) < 1e-5

mad_anomaly = calc_cleanse_mad(weights["l1_norms"])
assert torch.max(torch.abs(mad_anomaly - weights["mad_anomaly_ref"])) < 1e-5

h_benign = calc_strip(weights["logits_benign"])
assert torch.max(torch.abs(h_benign - weights["entropy_benign_ref"])) < 1e-5

h_trojan = calc_strip(weights["logits_trojan"])
assert torch.max(torch.abs(h_trojan - weights["entropy_trojan_ref"])) < 1e-5

x_tr = apply_trigger(weights["x_input"], weights["mask"], weights["pattern"])
assert torch.max(torch.abs(x_tr - weights["x_trojan_ref"])) < 1e-5
print("All Spectral, Neural Cleanse, STRIP, and BadNets assertions passed!")

In [ ]:
# Benchmarking trojan detection operators
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_spectral(weights["representations"])
    _ = calc_cleanse_mad(weights["l1_norms"])
    _ = calc_strip(weights["logits_benign"])
    _ = apply_trigger(weights["x_input"], weights["mask"], weights["pattern"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Combined trojan detection & trigger latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Undetected Backdoor Trigger vs Multi-Tier Defense
# 1. Neural Cleanse Anomaly: identifies infected class 2 as an outlier (anomaly > 2.0)
infected_class = torch.argmax(mad_anomaly).item()
max_anomaly = mad_anomaly[infected_class].item()
print(f"Neural Cleanse: Detected infected Class {infected_class} with MAD Anomaly Index = {max_anomaly:.2f} (> 2.0 threshold)")
assert max_anomaly > 2.0, "Neural Cleanse must flag infected class with anomaly > 2.0!"

# 2. STRIP Runtime: contrast benign entropy vs trojan entropy
h_b_val = h_benign.item()
h_t_val = h_trojan.item()
print(f"STRIP Entropy: Benign Input = {h_b_val:.4f} vs Trojan Triggered Input = {h_t_val:.4e}")
assert h_b_val > 1.0, "Clean inputs under superposition must yield high entropy!"
assert h_t_val < 1e-3, "Trojan inputs under superposition must yield near-zero entropy!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Spectral Signatures** isolate poisoned training samples in $\mathcal{O}(N \cdot d)$ via covariance SVD projections.
2. **Neural Cleanse** reverse-engineers candidate triggers across classes, flagging backdoors via Median Absolute Deviation (MAD).
3. **STRIP** detects test-time triggers in $<5\text{ms}$ by measuring Shannon entropy invariance under benign superposition.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.md](../../markdown/11_ai_safety_privacy_and_verification/06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.md)
- **PyTorch 2.x Module:** [06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.py](../../pytorch2/11_ai_safety_privacy_and_verification/06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.py)
- **Functional JAX Module:** [06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.py](../../jax/11_ai_safety_privacy_and_verification/06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.py)
- **Native Rust Candle Track:** [06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.rs](../../candle/11_ai_safety_privacy_and_verification/06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.rs)
- **Hardware Kernel (CPU / WGSL):** [06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.rs](../../kernels/11_ai_safety_privacy_and_verification/06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.rs)
- **Golden SafeTensors:** [06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.safetensors](../../weights/11_ai_safety_privacy_and_verification/06_backdoor_trojan_and_poisoning_defense_neural_cleanse_and_spectral.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 05. Safe RL Constrained MDP PPO Lagrangian CVaR and CBF](05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.ipynb) | [Master Index](../../README.md#11_ai_safety_privacy_and_verification) | [Next: 07. Privacy DP SGD PATE Gradient Inversion DLG and MIA](07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.ipynb) &rarr;